# IMPORTS

In [1]:
import pandas as pd
import numpy as np
import re
from collections import Counter

In [2]:
DATA_PATH = "/content/train_split.csv"

# 1. DATASET LOADING & SCHEMA INSPECTION

In [3]:
def load_dataset(filepath):
    """Load the AG News dataset and perform initial inspection."""
    df = pd.read_csv(filepath)
    print("=" * 70)
    print("1. DATASET LOADING & SCHEMA INSPECTION")
    print("=" * 70)
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nFirst 5 rows:\n{df.head()}")
    print(f"\nSample values per column:")
    for col in df.columns:
        print(f"  {col}: {df[col].head(5).tolist()}")
    return df

In [4]:
df = load_dataset(DATA_PATH)

1. DATASET LOADING & SCHEMA INSPECTION
Shape: (89320, 2)
Columns: ['text', 'label']

Dtypes:
text     object
label     int64
dtype: object

First 5 rows:
                                                text  label
0  A stinker? Critics pan Spacey #39;s Old Vic de...      4
1   quot;Hobbit quot; Joins Human Family A new an...      4
2  Merck stops \$2.5 bn painkiller Those taking V...      3
3  Six more die in Thai violence BANGKOK: Six mor...      1
4  Virgin launches online music service Virgin Di...      4

Sample values per column:
  text: ['A stinker? Critics pan Spacey #39;s Old Vic debut When Sir Laurence Olivier made the Old Vic in London the base of the embryonic National Theatre, he opened with Hamlet. Four decades later, Kevin Spacey, the film star, selected ', ' quot;Hobbit quot; Joins Human Family A new and tiny species of human that lived in Indonesia at the same time our own ancestors were colonising the world were discovered by scientists.', 'Merck stops \\$2.5 bn painki

# 2. DATA QUALITY ASSESSMENT

In [5]:
def assess_data_quality(df):
    """
    Comprehensive data quality assessment including:
    - Missing values
    - Duplicates
    - Text content anomalies (HTML entities, URLs, source tags, etc.)
    """
    print("\n" + "=" * 70)
    print("2. DATA QUALITY ASSESSMENT")
    print("=" * 70)

    # --- Completeness ---
    print("\n--- Completeness ---")
    print(f"Total rows: {len(df)}")
    print(f"Missing values:\n{df.isnull().sum()}")

    # --- Uniqueness ---
    print("\n--- Uniqueness ---")
    print(f"Exact duplicate rows: {df.duplicated().sum()}")
    print(f"Duplicate texts: {df.duplicated(subset='text').sum()}")

    # --- Text Content Anomalies ---
    print("\n--- Text Content Anomalies ---")

    # HTML entities
    html_entity_count = df['text'].str.contains(r'#[0-9]+;|&[a-z]+;', regex=True).sum()
    print(f"Rows with HTML entities: {html_entity_count} ({html_entity_count/len(df)*100:.1f}%)")

    # Escaped dollar signs
    dollar_count = df['text'].str.contains(r'\\\\\$', regex=True).sum()
    print(f"Rows with escaped dollar signs: {dollar_count}")

    # Numbers
    has_numbers = df['text'].str.contains(r'\d+', regex=True).sum()
    print(f"Rows with numbers: {has_numbers} ({has_numbers/len(df)*100:.1f}%)")

    # Special characters
    has_special = df['text'].str.contains(r'[^a-zA-Z0-9\s]', regex=True).sum()
    print(f"Rows with special chars: {has_special} ({has_special/len(df)*100:.1f}%)")

    # URLs
    url_count = df['text'].str.contains(r'http|www|\.com|\.net|\.org', regex=True).sum()
    print(f"Rows with URLs: {url_count} ({url_count/len(df)*100:.1f}%)")

    # Source tags
    ap_count = df['text'].str.contains(r'\bAP\b', regex=True).sum()
    reuters_count = df['text'].str.contains(r'\bReuters\b', regex=True).sum()
    afp_count = df['text'].str.contains(r'\bAFP\b', regex=True).sum()
    print(f"Rows with AP: {ap_count} ({ap_count/len(df)*100:.1f}%)")
    print(f"Rows with Reuters: {reuters_count} ({reuters_count/len(df)*100:.1f}%)")
    print(f"Rows with AFP: {afp_count} ({afp_count/len(df)*100:.1f}%)")

    # #NAME? artifact
    name_pattern = df['text'].str.contains(r'#NAME\?', regex=True).sum()
    print(f"Rows with #NAME?: {name_pattern}")

    # --- HTML Entity Breakdown ---
    print("\n--- HTML Entity Breakdown ---")
    entity_types = Counter()
    for text in df['text']:
        for m in re.finditer(r'#[0-9]+;', text):
            entity_types[m.group()] += 1
        for m in re.finditer(r'&[a-z]+;', text):
            entity_types[m.group()] += 1

    for ent, cnt in entity_types.most_common(20):
        print(f"  {ent}: {cnt}")

    # --- Source Tag Distribution by Class ---
    print("\n--- Source Tag Distribution by Class ---")
    label_names = {1: 'World', 2: 'Sports', 3: 'Business', 4: 'Sci/Tech'}
    for label in sorted(df['label'].unique()):
        subset = df[df['label'] == label]
        ap = subset['text'].str.contains(r'\bAP\b', regex=True).sum()
        reu = subset['text'].str.contains(r'\bReuters\b', regex=True).sum()
        afp = subset['text'].str.contains(r'\bAFP\b', regex=True).sum()
        name_l = label_names.get(label, label)
        print(f"  {name_l} (Label {label}): AP={ap} ({ap/len(subset)*100:.1f}%), "
              f"Reuters={reu} ({reu/len(subset)*100:.1f}%), AFP={afp} ({afp/len(subset)*100:.1f}%)")

    return {
        'html_entity_count': html_entity_count,
        'url_count': url_count,
        'ap_count': ap_count,
        'reuters_count': reuters_count,
        'afp_count': afp_count,
        'name_artifact_count': name_pattern,
        'entity_breakdown': entity_types,
    }

In [6]:
quality_metrics = assess_data_quality(df)


2. DATA QUALITY ASSESSMENT

--- Completeness ---
Total rows: 89320
Missing values:
text     0
label    0
dtype: int64

--- Uniqueness ---
Exact duplicate rows: 0
Duplicate texts: 0

--- Text Content Anomalies ---
Rows with HTML entities: 26405 (29.6%)
Rows with escaped dollar signs: 2
Rows with numbers: 51881 (58.1%)
Rows with special chars: 89193 (99.9%)
Rows with URLs: 2787 (3.1%)
Rows with AP: 6377 (7.1%)
Rows with Reuters: 9594 (10.7%)
Rows with AFP: 1923 (2.2%)
Rows with #NAME?: 20

--- HTML Entity Breakdown ---
  #39;: 33036
  &lt;: 9802
  &gt;: 9802
  #36;: 957
  #151;: 548
  #146;: 97
  #147;: 51
  #148;: 51
  #8217;: 24
  #38;: 18
  #8212;: 14
  #8221;: 10
  #145;: 9
  #8220;: 9
  #038;: 8
  #133;: 7
  #160;: 6
  #233;: 4
  #163;: 4
  #8211;: 4

--- Source Tag Distribution by Class ---
  World (Label 1): AP=2275 (10.2%), Reuters=2912 (13.0%), AFP=1385 (6.2%)
  Sports (Label 2): AP=2384 (10.7%), Reuters=1084 (4.9%), AFP=118 (0.5%)
  Business (Label 3): AP=207 (0.9%), Reuters=4

# 3. CLASS DISTRIBUTION ANALYSIS

In [7]:
def analyze_class_distribution(df):
    """Analyze and report class distribution."""
    print("\n" + "=" * 70)
    print("3. CLASS DISTRIBUTION ANALYSIS")
    print("=" * 70)

    label_names = {1: 'World', 2: 'Sports', 3: 'Business', 4: 'Sci/Tech'}
    print("Label distribution:")
    print(df['label'].value_counts().sort_index())
    print("\nLabel proportions:")
    print(df['label'].value_counts(normalize=True).sort_index().round(4))

    for label in sorted(df['label'].unique()):
        cnt = len(df[df['label'] == label])
        print(f"  {label_names[label]}: {cnt:,} samples ({cnt/len(df)*100:.1f}%)")

    return df['label'].value_counts().sort_index()

In [8]:
class_dist = analyze_class_distribution(df)


3. CLASS DISTRIBUTION ANALYSIS
Label distribution:
label
1    22330
2    22330
3    22330
4    22330
Name: count, dtype: int64

Label proportions:
label
1    0.25
2    0.25
3    0.25
4    0.25
Name: proportion, dtype: float64
  World: 22,330 samples (25.0%)
  Sports: 22,330 samples (25.0%)
  Business: 22,330 samples (25.0%)
  Sci/Tech: 22,330 samples (25.0%)


# 4. TEXT LENGTH & STATISTICAL ANALYSIS

In [9]:
def analyze_text_length(df):
    """
    Comprehensive text length analysis including:
    - Character length and word count statistics
    - Per-class length distributions
    - Percentile analysis for sequence length configuration
    - Outlier detection
    """
    print("\n" + "=" * 70)
    print("4. TEXT LENGTH & STATISTICAL ANALYSIS")
    print("=" * 70)

    df = df.copy()
    df['char_len'] = df['text'].str.len()
    df['word_count'] = df['text'].str.split().str.len()

    # Overall stats
    print("\n--- Overall Character Length Stats ---")
    print(df['char_len'].describe())

    print("\n--- Overall Word Count Stats ---")
    print(df['word_count'].describe())

    # Percentile analysis
    print("\n--- Word Count Percentile Analysis ---")
    percentiles = [50, 75, 90, 95, 99, 100]
    pct_vals = np.percentile(df['word_count'].values, percentiles)
    for p, v in zip(percentiles, pct_vals):
        print(f"  {p}th percentile: {int(v)} words")

    # Per-class stats
    label_names = {1: 'World', 2: 'Sports', 3: 'Business', 4: 'Sci/Tech'}
    print("\n--- Per-Class Text Length Stats ---")
    for label in sorted(df['label'].unique()):
        subset = df[df['label'] == label]
        print(f"  {label_names[label]} (Label {label}): "
              f"avg_words={subset['word_count'].mean():.1f}, "
              f"avg_chars={subset['char_len'].mean():.1f}, "
              f"median_words={subset['word_count'].median():.1f}")

    # Outlier detection
    short_texts = df[df['word_count'] < 10]
    long_texts = df[df['word_count'] > 80]
    print(f"\nTexts with <10 words: {len(short_texts)}")
    print(f"Texts with >80 words: {len(long_texts)}")

    if len(short_texts) > 0:
        print("\nSample short texts:")
        for _, row in short_texts.head(3).iterrows():
            print(f"  [{row['label']}] ({row['word_count']}w): {row['text'][:200]}")

    # Capitalization ratio
    df['caps_ratio'] = df['text'].apply(
        lambda x: sum(1 for c in x if c.isupper()) / len(x) if len(x) > 0 else 0
    )
    print("\n--- Capitalization Ratio ---")
    print(df['caps_ratio'].describe())
    for label in sorted(df['label'].unique()):
        ratio = df[df['label'] == label]['caps_ratio'].mean()
        print(f"  {label_names[label]}: avg caps ratio = {ratio:.3f}")

    return df

In [10]:
df_with_lengths = analyze_text_length(df)


4. TEXT LENGTH & STATISTICAL ANALYSIS

--- Overall Character Length Stats ---
count    89320.000000
mean       236.464196
std         66.489560
min         17.000000
25%        196.000000
50%        232.000000
75%        266.000000
max       1012.000000
Name: char_len, dtype: float64

--- Overall Word Count Stats ---
count    89320.000000
mean        37.861722
std         10.112535
min          4.000000
25%         32.000000
50%         37.000000
75%         43.000000
max        177.000000
Name: word_count, dtype: float64

--- Word Count Percentile Analysis ---
  50th percentile: 37 words
  75th percentile: 43 words
  90th percentile: 48 words
  95th percentile: 53 words
  99th percentile: 70 words
  100th percentile: 177 words

--- Per-Class Text Length Stats ---
  World (Label 1): avg_words=38.9, avg_chars=242.6, median_words=39.0
  Sports (Label 2): avg_words=37.8, avg_chars=224.9, median_words=37.0
  Business (Label 3): avg_words=37.5, avg_chars=241.0, median_words=37.0
  Sci/Tech

# 5. VOCABULARY & LINGUISTIC ANALYSIS

In [11]:
def analyze_vocabulary(df, use_stopwords=True):
    """
    Vocabulary and linguistic analysis including:
    - Total and unique token counts
    - Per-class vocabulary comparison
    - Top unigrams, bigrams, and trigrams by class
    - Vocabulary overlap (Jaccard similarity)
    """
    print("\n" + "=" * 70)
    print("5. VOCABULARY & LINGUISTIC ANALYSIS")
    print("=" * 70)

    label_names = {1: 'World', 2: 'Sports', 3: 'Business', 4: 'Sci/Tech'}

    try:
        from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
        stopwords = ENGLISH_STOP_WORDS if use_stopwords else set()
    except ImportError:
        stopwords = set()

    # --- Overall Vocabulary ---
    all_words = ' '.join(df['text'].astype(str).tolist()).lower().split()
    unique_words = set(all_words)
    print(f"\nTotal tokens (lowercase split): {len(all_words):,}")
    print(f"Unique tokens: {len(unique_words):,}")
    print(f"Type-Token Ratio: {len(unique_words)/len(all_words):.4f}")

    # --- Per-Class Vocabulary ---
    print("\n--- Per-Class Vocabulary ---")
    class_vocab = {}
    class_token_counts = {}

    for label in sorted(df['label'].unique()):
        # With stopwords
        subset_words = ' '.join(df[df['label'] == label]['text'].astype(str).tolist()).lower().split()
        unique = set(subset_words)
        class_vocab[label] = unique
        print(f"  {label_names[label]}: {len(subset_words):,} tokens, {len(unique):,} unique (all)")

        # Without stopwords (for cleaner analysis)
        filtered = [t for t in subset_words if t not in stopwords and len(t) > 2]
        class_token_counts[label] = len(filtered)
        unique_filtered = set(filtered)
        print(f"  {label_names[label]}: {len(filtered):,} tokens, {len(unique_filtered):,} unique (filtered)")

    # --- Jaccard Similarity (Vocabulary Overlap) ---
    print("\n--- Jaccard Similarity Between Class Pairs ---")
    pairs = [(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]
    for a, b in pairs:
        inter = len(class_vocab[a] & class_vocab[b])
        union = len(class_vocab[a] | class_vocab[b])
        jaccard = inter / union if union > 0 else 0
        print(f"  {label_names[a]} vs {label_names[b]}: {jaccard:.4f}")

    # --- Top Unigrams per Class ---
    print("\n--- Top Unigrams per Class ---")
    for label in sorted(df['label'].unique()):
        subset = df[df['label'] == label]['text'].tolist()
        counter = Counter()
        for text in subset:
            tokens = re.findall(r'[a-z]+', text.lower())
            if use_stopwords:
                tokens = [t for t in tokens if t not in stopwords and len(t) > 2]
            counter.update(tokens)
        print(f"\n  {label_names[label]} (Label {label}):")
        for word, cnt in counter.most_common(20):
            print(f"    {word}: {cnt}")

    # --- Top Bigrams per Class ---
    print("\n--- Top Bigrams per Class ---")
    for label in sorted(df['label'].unique()):
        subset = df[df['label'] == label]['text'].tolist()
        counter = Counter()
        for text in subset:
            tokens = re.findall(r'[a-z]+', text.lower())
            if use_stopwords:
                tokens = [t for t in tokens if t not in stopwords and len(t) > 2]
            for i in range(len(tokens) - 1):
                counter[(tokens[i], tokens[i + 1])] += 1
        print(f"\n  {label_names[label]} (Label {label}):")
        for bg, cnt in counter.most_common(15):
            print(f"    {' '.join(bg)}: {cnt}")

    # --- Top Trigrams per Class ---
    print("\n--- Top Trigrams per Class ---")
    for label in sorted(df['label'].unique()):
        subset = df[df['label'] == label]['text'].tolist()
        counter = Counter()
        for text in subset:
            tokens = re.findall(r'[a-z]+', text.lower())
            if use_stopwords:
                tokens = [t for t in tokens if t not in stopwords and len(t) > 2]
            for i in range(len(tokens) - 2):
                counter[(tokens[i], tokens[i + 1], tokens[i + 2])] += 1
        print(f"\n  {label_names[label]} (Label {label}):")
        for tg, cnt in counter.most_common(10):
            print(f"    {' '.join(tg)}: {cnt}")

In [12]:
analyze_vocabulary(df, use_stopwords=True)


5. VOCABULARY & LINGUISTIC ANALYSIS

Total tokens (lowercase split): 3,381,809
Unique tokens: 135,689
Type-Token Ratio: 0.0401

--- Per-Class Vocabulary ---
  World: 868,467 tokens, 49,058 unique (all)
  World: 550,805 tokens, 48,408 unique (filtered)
  Sports: 844,780 tokens, 50,487 unique (all)
  Sports: 522,153 tokens, 49,722 unique (filtered)
  Business: 837,705 tokens, 48,179 unique (all)
  Business: 551,448 tokens, 47,444 unique (filtered)
  Sci/Tech: 830,857 tokens, 59,191 unique (all)
  Sci/Tech: 528,281 tokens, 58,380 unique (filtered)

--- Jaccard Similarity Between Class Pairs ---
  World vs Sports: 0.2322
  World vs Business: 0.2400
  World vs Sci/Tech: 0.2287
  Sports vs Business: 0.2059
  Sports vs Sci/Tech: 0.1979
  Business vs Sci/Tech: 0.2495

--- Top Unigrams per Class ---

  World (Label 1):
    said: 5869
    iraq: 4362
    reuters: 3990
    president: 3242
    new: 2662
    afp: 2549
    minister: 2484
    killed: 2306
    people: 2110
    bush: 1981
    governmen

# 6. DATA QUALITY DEEP DIVE

In [13]:
def deep_dive_quality_issues(df):
    """
    Detailed analysis of data quality issues with per-class breakdowns.
    Focuses on HTML entities, source tags, and URL artifacts.
    """
    print("\n" + "=" * 70)
    print("6. DATA QUALITY DEEP DIVE")
    print("=" * 70)

    label_names = {1: 'World', 2: 'Sports', 3: 'Business', 4: 'Sci/Tech'}

    # --- Data Quality Issue Prevalence Matrix ---
    print("\n--- Data Quality Issue Prevalence by Class (%) ---")
    issues = {
        'HTML Entities': [],
        'URLs/Links': [],
        'Source Tags (AP/Reuters/AFP)': [],
        'Special Characters': [],
        'Numbers': [],
        'Escaped $': [],
    }

    for label in sorted(df['label'].unique()):
        subset = df[df['label'] == label]
        issues['HTML Entities'].append(subset['text'].str.contains(r'#[0-9]+;|&[a-z]+;', regex=True).mean() * 100)
        issues['URLs/Links'].append(subset['text'].str.contains(r'http|www|\.com|\.net|\.org', regex=True).mean() * 100)
        issues['Source Tags (AP/Reuters/AFP)'].append(subset['text'].str.contains(r'\b(AP|Reuters|AFP)\b', regex=True).mean() * 100)
        issues['Special Characters'].append(subset['text'].str.contains(r'[^a-zA-Z0-9\s]', regex=True).mean() * 100)
        issues['Numbers'].append(subset['text'].str.contains(r'\d+', regex=True).mean() * 100)
        issues['Escaped $'].append(subset['text'].str.contains(r'\\\\\$', regex=True).mean() * 100)

    # Print as matrix
    header = f"{'Issue':<30} {'World':>8} {'Sports':>8} {'Business':>8} {'Sci/Tech':>8}"
    print(header)
    print("-" * len(header))
    for issue_name, vals in issues.items():
        row = f"{issue_name:<30}"
        for v in vals:
            row += f" {v:>7.1f}%"
        print(row)

In [14]:
deep_dive_quality_issues(df)


6. DATA QUALITY DEEP DIVE

--- Data Quality Issue Prevalence by Class (%) ---


/tmp/ipykernel_16400/2770792608.py:27: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  issues['Source Tags (AP/Reuters/AFP)'].append(subset['text'].str.contains(r'\b(AP|Reuters|AFP)\b', regex=True).mean() * 100)


Issue                             World   Sports Business Sci/Tech
------------------------------------------------------------------
HTML Entities                     22.8%    34.6%    34.9%    26.0%
URLs/Links                         0.3%     0.5%     6.3%     5.4%
Source Tags (AP/Reuters/AFP)      29.4%    16.1%    20.5%    14.1%
Special Characters                99.7%    99.9%    99.9%    99.9%
Numbers                           47.6%    70.0%    63.8%    51.0%
Escaped $                          0.0%     0.0%     0.0%     0.0%


# 7. PER-CLASS FEATURE PROFILING

In [15]:
def profile_classes(df):
    """
    Generate comprehensive per-class profiles combining
    text length, vocabulary, and quality metrics.
    """
    print("\n" + "=" * 70)
    print("7. PER-CLASS FEATURE PROFILING")
    print("=" * 70)

    label_names = {1: 'World', 2: 'Sports', 3: 'Business', 4: 'Sci/Tech'}

    try:
        from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
        stopwords = ENGLISH_STOP_WORDS
    except ImportError:
        stopwords = set()

    df = df.copy()
    df['word_count'] = df['text'].str.split().str.len()
    df['char_len'] = df['text'].str.len()

    for label in sorted(df['label'].unique()):
        subset = df[df['label'] == label]
        name = label_names[label]

        # Vocabulary
        vocab = set()
        for text in subset['text']:
            tokens = re.findall(r'[a-z]+', text.lower())
            tokens = [t for t in tokens if t not in stopwords and len(t) > 2]
            vocab.update(tokens)

        # Top keywords
        counter = Counter()
        for text in subset['text']:
            tokens = re.findall(r'[a-z]+', text.lower())
            tokens = [t for t in tokens if t not in stopwords and len(t) > 2]
            counter.update(tokens)
        top5 = [w for w, _ in counter.most_common(5)]

        print(f"\n{'='*40}")
        print(f"  {name} (Label {label})")
        print(f"{'='*40}")
        print(f"  Sample count:        {len(subset):,}")
        print(f"  Avg word count:      {subset['word_count'].mean():.1f}")
        print(f"  Median word count:   {subset['word_count'].median():.1f}")
        print(f"  Avg char length:     {subset['char_len'].mean():.1f}")
        print(f"  Unique vocab (filt): {len(vocab):,}")
        print(f"  Top 5 keywords:      {', '.join(top5)}")

        # Source tag prevalence
        reuters_pct = subset['text'].str.contains(r'\bReuters\b', regex=True).mean() * 100
        ap_pct = subset['text'].str.contains(r'\bAP\b', regex=True).mean() * 100
        afp_pct = subset['text'].str.contains(r'\bAFP\b', regex=True).mean() * 100
        html_pct = subset['text'].str.contains(r'#[0-9]+;|&[a-z]+;', regex=True).mean() * 100
        url_pct = subset['text'].str.contains(r'http|www|\.com|\.net|\.org', regex=True).mean() * 100

        print(f"  Reuters prevalence:  {reuters_pct:.1f}%")
        print(f"  AP prevalence:       {ap_pct:.1f}%")
        print(f"  AFP prevalence:      {afp_pct:.1f}%")
        print(f"  HTML entities:       {html_pct:.1f}%")
        print(f"  URL fragments:       {url_pct:.1f}%")

In [16]:
profile_classes(df)


7. PER-CLASS FEATURE PROFILING

  World (Label 1)
  Sample count:        22,330
  Avg word count:      38.9
  Median word count:   39.0
  Avg char length:     242.6
  Unique vocab (filt): 25,525
  Top 5 keywords:      said, iraq, reuters, president, new
  Reuters prevalence:  13.0%
  AP prevalence:       10.2%
  AFP prevalence:      6.2%
  HTML entities:       22.8%
  URL fragments:       0.3%

  Sports (Label 2)
  Sample count:        22,330
  Avg word count:      37.8
  Median word count:   37.0
  Avg char length:     224.9
  Unique vocab (filt): 26,443
  Top 5 keywords:      game, new, season, team, win
  Reuters prevalence:  4.9%
  AP prevalence:       10.7%
  AFP prevalence:      0.5%
  HTML entities:       34.6%
  URL fragments:       0.5%

  Business (Label 3)
  Sample count:        22,330
  Avg word count:      37.5
  Median word count:   37.0
  Avg char length:     241.0
  Unique vocab (filt): 23,223
  Top 5 keywords:      reuters, said, new, oil, stocks
  Reuters prevalence: